# Universal Audio Transcriber (Faster-Whisper GPU)

Free and fast tool for transcribing audio and video files of any length (lectures, meetings, interviews, voice notes) using Google Colab hardware acceleration.

### Quick Start Guide:
1. **Check your GPU:** Make sure the **T4 GPU** accelerator is enabled in the top right (or under `Runtime` > `Change runtime type`).
2. **Upload:** Launch **Cell 1** and select the audio or video file from your computer using the `Choose file` button.
> 💡 *Very large files (>50 MB)?* Press `Cancel upload` on the button, open the **Files** panel (📁 icon in the left bar), and drag your file there: uploading will be much faster and the cell will detect it automatically.
3. **Transcription and download:** In **Cell 2**, set your desired options (language, timestamp, and any audio-specific keywords) and launch the cell. Once complete, the .txt file will automatically download to your PC's **Downloads** folder.

In [ ]:
# @title 1. Environment installation and audio/video file upload
import os
import torch
from google.colab import files

# Check GPU availability
if not torch.cuda.is_available():
    print("⚠️ WARNING: GPU is not active! Go to 'Runtime > Change runtime type' and select free 'T4 GPU' for faster transcription.")
else:
    print(f"✅ Active GPU: {torch.cuda.get_device_name(0)}")

# Install faster-whisper
print("Installing and configuring faster-whisper on Google machine...")
!pip install -q -U faster-whisper nvidia-cublas-cu12 nvidia-cudnn-cu12

# Select file from PC
SUPPORTED_EXTENSIONS = (
    ".mp3", ".wav", ".m4a", ".aac", ".ogg", ".opus", ".flac", ".wma",
    ".mp4", ".mkv", ".mov", ".avi", ".webm"
)

print("\n📂 Select an audio/video file from your PC (or press 'Cancel upload' if you dragged it into the left sidebar):")
uploaded = files.upload()

if uploaded:
    audio_filename = list(uploaded.keys())[0]
    print(f"\n✅ File successfully uploaded: '{audio_filename}'")
else:
    local_files = [
        f for f in os.listdir(".")
        if f.lower().endswith(SUPPORTED_EXTENSIONS) and not f.startswith("clean_")
    ]
    if local_files:
        audio_filename = max(local_files, key=os.path.getmtime)
        print(f"\n✅ Detected file from local storage: '{audio_filename}'")
    else:
        raise FileNotFoundError("❌ No audio/video file selected or found. Please run the cell again.")

In [ ]:
# @title 2. Pre-processing, Transcription and .txt Download
import os
import glob
import time
import wave
import ctypes
import logging
import subprocess
import warnings
import numpy as np
import torch
from google.colab import files

# Removing WARNING
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")
for logger_name in ["huggingface_hub", "huggingface_hub.utils._http"]:
    log = logging.getLogger(logger_name)
    log.setLevel(logging.ERROR)
    log.propagate = False

# Pre-load CUDA 12
def preload_cuda_libs():
    site_packages = "/usr/local/lib/python*/dist-packages/nvidia/*/lib"
    lib_dirs = glob.glob(site_packages)
    target_libs = [
        "libcublasLt.so.12",
        "libcublas.so.12",
        "libcudnn.so.9",
        "libcudnn_ops.so.9",
        "libcudnn_cnn.so.9"
    ]
    for lib_name in target_libs:
        for d in lib_dirs:
            lib_path = os.path.join(d, lib_name)
            if os.path.exists(lib_path):
                try:
                    ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass
preload_cuda_libs()


from faster_whisper import WhisperModel, BatchedInferencePipeline

# User interface
# @markdown ### ⚙️ Transcription Settings
language = "Auto-detect" # @param ["Auto-detect", "en", "it", "es", "fr", "de"]
model_size = "large-v3-turbo" # @param ["large-v3-turbo", "large-v3", "medium"]
include_timestamps = False # @param {type:"boolean"}
# @markdown ---
# @markdown ### 💡 Custom Vocabulary *(Optional)*
# @markdown *Leave empty for general audio. If the audio contains proper nouns, acronyms, or technical jargon, enter them separated by commas (e.g., `John Doe, Kubernetes, EBITDA, 5ms`):*
optional_keywords = "" # @param {type:"string"}

if "audio_filename" not in locals() or not os.path.exists(audio_filename):
    raise RuntimeError("❌ File not found! Please run Cell 1 first to upload the audio.")

base_name = os.path.splitext(os.path.basename(audio_filename))[0]
clean_audio = f"clean_{base_name}.wav"
output_txt = f"{base_name}_transcription.txt"

# Pre-processing with FFmpeg (16kHz, Mono, PCM WAV, Highpass 80Hz + dynamic audio normalizer)
print(f"Optimizing and normalizing audio volume for '{audio_filename}'...")
t0 = time.time()
ffmpeg_cmd = [
    "ffmpeg", "-y", "-i", audio_filename,
    "-vn", "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le",
    "-af", "highpass=f=80,dynaudnorm=p=0.9:s=5",
    clean_audio
]
subprocess.run(ffmpeg_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, check=True)

# Load 16-bit PCM directly into a float32 NumPy array
with wave.open(clean_audio, "rb") as wf:
    audio_array = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0

print(f"✅ Audio ready in {time.time() - t0:.1f}s")

# Load Faster-Whisper model
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"🚀 Loading model '{model_size}' ({compute_type}) on {device.upper()}...")
model = WhisperModel(model_size, device=device, compute_type=compute_type)
batched_model = BatchedInferencePipeline(model=model)

# Dynamic handling of language and optional vocabulary
lang_param = None if language == "Auto-detect" else language
clean_keywords = optional_keywords.strip()

hotwords_param = clean_keywords if clean_keywords else None
prompt_param = clean_keywords if clean_keywords else None

print("Starting transcription:\n" + "-"*60)
t_transcribe = time.time()

segments, info = batched_model.transcribe(
    audio_array,
    batch_size=16,
    language=lang_param,
    initial_prompt=prompt_param,
    hotwords=hotwords_param,
    condition_on_previous_text=False,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500, speech_pad_ms=200)
)

if lang_param is None:
    print(f"Automatically detected language: {info.language.upper()} (confidence: {info.language_probability:.0%})\n" + "-"*60)

def format_ts(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# Write to file and stream output in real time
with open(output_txt, "w", encoding="utf-8") as f:
    for segment in segments:
        text = segment.text.strip()
        if not text:
            continue
        ts_str = f"[{format_ts(segment.start)} -> {format_ts(segment.end)}] "
        print(f"{ts_str}{text}")
        
        if include_timestamps:
            f.write(f"{ts_str}{text}\n")
        else:
            f.write(f"{text}\n")

elapsed = time.time() - t_transcribe
print("-" * 60)
print(f"✅ Completed in {elapsed/60:.1f} min (Audio duration: {info.duration/60:.1f} min | Speed: {info.duration/elapsed:.1f}x)")

# Clean up temporary files and trigger download
if os.path.exists(clean_audio):
    os.remove(clean_audio)

print(f"⬇️ Downloading '{output_txt}' to your PC's Downloads folder...")
files.download(output_txt)